# NLP Robustness Study — SST-2
**Team:** Curtis Cao, Xilu Zeng, Soham Agarwal, Ian Abusamra

This notebook is the Colab entry point. It installs dependencies, downloads GloVe, then runs the full experiment pipeline.

---
**Sections**
1. Setup (install deps, mount Drive)
2. Download GloVe embeddings
3. Generate, verify, and save perturbations
4. Train baseline models
5. Run full experiment sweep
6. Analyze & visualize results

## 1. Setup

In [1]:
!pip install -q datasets transformers torch scikit-learn seaborn tqdm joblib

In [2]:
# Mount Google Drive (skip if using a university cluster)
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [4]:
import os
from pathlib import Path

# Option A: clone from GitHub (recommended)
#!git clone https://github.com/CURT1S03/NLPProject.git /content/NLPProject
#PROJECT_ROOT = '/content/NLPProject'

# Option B: project already in Drive
PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

Working directory: /Users/ianabusamra/Desktop/NLPProject


## 2. Download GloVe Embeddings
Only needed for the NBOW model. Skipped on subsequent runs if the file already exists.

In [6]:
import os
import urllib.request
import zipfile

GLOVE_DIR = os.path.join(PROJECT_ROOT, "data")
GLOVE_FILE = os.path.join(GLOVE_DIR, "glove.6B.100d.txt")
ZIP_FILE = os.path.join(GLOVE_DIR, "glove.6B.zip")

os.makedirs(GLOVE_DIR, exist_ok=True)

if not os.path.exists(GLOVE_FILE):
    print("Downloading GloVe 6B 100d (~330 MB) ...")
    urllib.request.urlretrieve(
        "https://nlp.stanford.edu/data/glove.6B.zip",
        ZIP_FILE
    )

    print("Extracting glove.6B.100d.txt ...")
    with zipfile.ZipFile(ZIP_FILE, "r") as z:
        z.extract("glove.6B.100d.txt", GLOVE_DIR)

    print("Done.")
else:
    print(f"GloVe already exists at {GLOVE_FILE}")

Extracting glove.6B.100d.txt ...
Done.


## 3. Generate, Verify, and Save Perturbations
Run the full perturbation suite on the SST-2 validation split, preview examples for every condition, and save the reusable CSV to `results/perturbations/validation_perturbations.csv`.

In [8]:
!python -m data.perturbations

Saved 20056 rows to /Users/ianabusamra/Desktop/NLPProject/results/perturbations/validation_perturbations.csv


In [ ]:
# Inspect the saved perturbation CSV that later steps can reuse
import pandas as pd

perturb_df = pd.read_csv('results/perturbations/validation_perturbations.csv')
print('Rows:', len(perturb_df))
print('Unique conditions:')
print(perturb_df[['perturbation_type', 'severity']].drop_duplicates().to_string(index=False))
perturb_df.head(12)

## 4. Train Baseline Models
Trains LR and NBOW on the SST-2 training split; caches models to `results/`.
DistilBERT uses a pre-fine-tuned checkpoint — no training needed.

In [ ]:
# Logistic Regression + TF-IDF (~30s)
!python models/baseline_lr.py

In [ ]:
# NBOW: GloVe load + 10 epochs MLP (~5-10 min on GPU)
!python models/baseline_nbow.py

In [ ]:
# DistilBERT: download checkpoint + verify clean accuracy
!python models/bert_eval.py

## 5. Run Full Experiment Sweep
Evaluates 3 models × 13 conditions → `results/results.csv` (39 rows).
Expected runtime on Colab GPU: ~20–30 minutes.

In [ ]:
!python experiments/run_experiments.py

In [ ]:
# Preview the results table
import pandas as pd
df = pd.read_csv('results/results.csv')
print(df.to_string(index=False))

## 6. Analyze & Visualize Results

In [ ]:
!python experiments/analyze_results.py

In [ ]:
from IPython.display import Image, display
display(Image('results/plots/accuracy_curves.png'))
display(Image('results/plots/drop_heatmap.png'))